# Linear regression via OLS, GD, and pytorch native

In [299]:
import numpy as np
import torch
from random import shuffle
import math

### 0. Functions

In [300]:
def calculate_ols_beta(X, y):
    m = np.linalg.inv(np.matmul(np.transpose(X), X))
    m = np.matmul(m, np.transpose(X))
    m = np.dot(m, y)
    return m

In [301]:
def calculate_gd_beta(X, y, shape, max_iter=1000, threshold=1e-8, learning_rate = 0.01):
    gd_beta = np.zeros(shape=shape)
    loss = float('inf')
    iter=0
    
    while (iter < max_iter) and (loss > threshold):
        y_pred = np.matmul(X, gd_beta)
        error = y_pred - y
        loss = np.mean(error**2)

        # Gradient of MSE has the 1/n normalization to avoid explosions
        grad_beta = (- 2 * y.T @ X + 2 * gd_beta @ X.T @ X).T / len(y)
        # Better is 
        # grad_beta = (2/len(y)) (X_design.T @ error )
        gd_beta = gd_beta - learning_rate * grad_beta
        iter +=1

    return gd_beta


In [302]:
def mse(y, y_pred):
    error = y_pred - y
    return(np.mean(error**2))

In [ ]:
def calculate_sgd_beta(X, y, max_epochs, model, batch_size, threshold = 1e-8, lr = 0.01):
    # Initialization of loss, epoch, criterion
    loss = float('inf')
    epoch=0
    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # We do not drop any elements
    n_batches = math.ceil(X.shape[0] / batch_size)

    # We use a while loop to be able to add a stopping criterion on loss, in case
    while (epoch < max_epochs):
        # initialize batch
        perm = torch.randperm(X.shape[0])
        X = X[perm]
        y = y[perm]
        
        for i in range(n_batches):
            # End index handles size of last batch
            start_idx = i *batch_size
            end_idx   = min((i+1) * batch_size, X.shape[0])

            # Extract batch X and y
            X_batch = X[start_idx:end_idx,:]
            y_batch = y[start_idx:end_idx, :].view(-1, 1)
        
            # Run the GD part
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        epoch +=1
    print(f"reached epoch: {epoch}")    
    return model.weight.detach().clone().view(-1, 1)

### 0. Basic data

In [304]:
n = 100 # Number of observations
d = 3 # Number of dimensions

w_true = np.array([1.2, -1.3, 2.15])
b_true = np.array([0.37])

beta_true = np.concatenate((b_true, w_true))

X = np.random.uniform(low=-10, high=10, size=(n, d))

X_design = np.c_[np.ones(X.shape[0]), X]

X_mean = X_design[:, 1:].mean(axis=0)
X_std = X_design[:, 1:].std(axis=0)

beta_true_norm = np.concatenate((b_true + np.dot(X_mean, w_true), w_true * X_std))

X_design[:, 1:] = (X_design[:, 1:] - X_mean) / X_std

### 1. Generate random "reasonable" data

In [305]:
noise = 0.1 * np.random.normal(size=(n, ))
y = np.dot(X, w_true) + b_true + noise



In [306]:
# OLS solution
beta_ols = calculate_ols_beta(X_design, y)

# GD solution
beta_gd = calculate_gd_beta(X_design, y, beta_true.shape, 1000, 1e-8, 0.01)

# Compare with pytorch
X_design_tensor = torch.tensor(X_design, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)

# Torch OLS
beta_torch_ols = torch.linalg.lstsq(X_design_tensor, y_tensor)

# Torch SGD
model = torch.nn.Linear(d+1, 1, bias=False)
beta_torch_sgd = calculate_sgd_beta(X_design_tensor, y_tensor, 1000, model, 32, lr=0.03)


# Compare
print(f"True beta: {beta_true_norm}")
print(f"OLS beta: {beta_ols}")
print(f"GD beta: {beta_gd}")

print(f"pytorch OLS beta: {np.array(beta_torch_ols.solution).T[0]}")
print(f"pytorch OLS SGD: {np.array(beta_torch_sgd).T[0]}")


reached epoch: 1000
True beta: [ 0.25886932  7.04979314 -7.17754867 11.78222753]
OLS beta: [ 0.25909069  7.03349181 -7.1674064  11.78276907]
GD beta: [ 0.25909069  7.0334918  -7.16740636 11.78276903]
pytorch OLS beta: [ 0.25909072  7.033492   -7.167408   11.78277   ]
pytorch OLS SGD: [ 0.2631902  7.0340543 -7.1758103 11.77593  ]


C:\Users\MATTROSS\AppData\Local\Temp\ipykernel_26144\1043974909.py:24: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print(f"pytorch OLS beta: {np.array(beta_torch_ols.solution).T[0]}")
C:\Users\MATTROSS\AppData\Local\Temp\ipykernel_26144\1043974909.py:25: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print(f"pytorch OLS SGD: {np.array(beta_torch_sgd).T[0]}")


### 2. Extremely noisy data

In [310]:
noise = 10 * np.random.normal(size=(n, ))
y = np.dot(X, w_true) + b_true + noise


In [309]:
# OLS solution
beta_ols = calculate_ols_beta(X_design, y)

# GD solution
beta_gd = calculate_gd_beta(X_design, y, beta_true.shape, 1000, 1e-8, 0.01)

# Compare with pytorch
X_design_tensor = torch.tensor(X_design, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)

# Torch OLS
beta_torch_ols = torch.linalg.lstsq(X_design_tensor, y_tensor)

# Torch SGD
model = torch.nn.Linear(d+1, 1, bias=False)
beta_torch_sgd = calculate_sgd_beta(X_design_tensor, y_tensor, 1000, model, 32, lr=0.03)


# Compare
print(f"True beta: {beta_true_norm}")
print(f"OLS beta: {beta_ols}")
print(f"GD beta: {beta_gd}")

print(f"pytorch OLS beta: {np.array(beta_torch_ols.solution).T[0]}")
print(f"pytorch OLS SGD: {np.array(beta_torch_sgd).T[0]}")


reached epoch: 1000
True beta: [ 0.25886932  7.04979314 -7.17754867 11.78222753]
OLS beta: [-0.9916887   6.06618987 -7.02403007 11.05090896]
GD beta: [-0.9916887   6.06618985 -7.02403003 11.05090891]
pytorch OLS beta: [-0.9916889  6.0661902 -7.0240297 11.050907 ]
pytorch OLS SGD: [-1.2774894  6.5002027 -7.046583  11.28344  ]


C:\Users\MATTROSS\AppData\Local\Temp\ipykernel_26144\1043974909.py:24: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print(f"pytorch OLS beta: {np.array(beta_torch_ols.solution).T[0]}")
C:\Users\MATTROSS\AppData\Local\Temp\ipykernel_26144\1043974909.py:25: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print(f"pytorch OLS SGD: {np.array(beta_torch_sgd).T[0]}")


### 3. Collinearity

In [311]:
X_coll = X[:,1] + X[:,0] + 0.01 * np.random.normal(size=(n,))
noise = 0.1 * np.random.normal(size=(n, ))

X_design_coll = np.c_[np.ones(X.shape[0]), X[:, :-1], X_coll]
y = X_design_coll @ beta_true + noise

In [312]:
# OLS solution
beta_ols = calculate_ols_beta(X_design_coll, y)

# GD solution
beta_gd = calculate_gd_beta(X_design_coll, y, beta_true.shape, 1000, 1e-8, 0.01)

# Compare with pytorch
X_design_tensor = torch.tensor(X_design_coll, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)

# Torch OLS
beta_torch_ols = torch.linalg.lstsq(X_design_tensor, y_tensor)

# Torch SGD
model = torch.nn.Linear(d+1, 1, bias=False)
beta_torch_sgd = calculate_sgd_beta(X_design_tensor, y_tensor, 1000, model, 32, lr=0.03)


# Compare
print(f"True beta: {beta_true_norm}")
print(f"OLS beta: {beta_ols}")
print(f"GD beta: {beta_gd}")

print(f"pytorch OLS beta: {np.array(beta_torch_ols.solution).T[0]}")
print(f"pytorch OLS SGD: {np.array(beta_torch_sgd).T[0]}")


reached epoch: 1000
True beta: [ 0.25886932  7.04979314 -7.17754867 11.78222753]
OLS beta: [ 0.37870778  0.6130724  -1.88654471  2.73563683]
GD beta: [ 0.37976924  1.9486538  -0.55116226  1.40058576]
pytorch OLS beta: [ 0.3787067  0.6131308 -1.8864862  2.735578 ]
pytorch OLS SGD: [nan nan nan nan]


C:\Users\MATTROSS\AppData\Local\Temp\ipykernel_26144\1357533555.py:24: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print(f"pytorch OLS beta: {np.array(beta_torch_ols.solution).T[0]}")
C:\Users\MATTROSS\AppData\Local\Temp\ipykernel_26144\1357533555.py:25: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  print(f"pytorch OLS SGD: {np.array(beta_torch_sgd).T[0]}")


In [313]:
y_ols = X_design_coll @ beta_ols
y_gd = X_design_coll @ beta_gd

mse_ols = mse(y, y_ols)
mse_gd  = mse(y, y_gd)

print(f"OLS mse: {mse_ols}")
print(f"GD mse: {mse_gd}")

OLS mse: 0.011339561561984218
GD mse: 0.011483349962503888
